In [5]:
from dotenv import load_dotenv

load_dotenv()

True

## 建立 Examples

In [3]:
pizza_order_follow_up = [
  {
    "inputs": {
      "question": "我要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。"
    },
    "outputs": {
      "answer": "請問訂單要登記什麼名字？",
      "requires_followup": True
    },
    "metadata": {
      "case_id": "PIZZA_FOLLOWUP_001",
      "missing_field": "customer_name"
    }
  }]


In [4]:
pizza_order_normal = [
  {
    "inputs": {
      "question": "我叫小陳，要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。"
    },
    "outputs": {
      "answer": "已確認您的訂單，小陳：一個小尺寸、手拋餅皮的原味起司 Pizza，自取，預計 15–20 分鐘後可取餐。",
      "requires_followup": False
    },
    "metadata": {
      "case_id": "PIZZA_NO_FOLLOWUP_001",
      "requires_followup": False
    }
  }]

## 上傳至 LangSmith

In [6]:
from langsmith import Client
from uuid import uuid4

examples = pizza_order_follow_up[0:2] + pizza_order_normal[0:2]
dataset_name = "pizza_order_followup" + str(uuid4())

client = Client()
dataset = client.create_dataset(dataset_name,
                                    description="測試 Pizza follow up 功能 "
                                    )
# Create examples
response = client.create_examples(dataset_id=dataset.id,
                           examples=examples)

print(f"成功上傳：{response['count']} 筆")
print(f"Example IDs：{response['example_ids']}")
print(f"資料版本時間：{response.get('as_of')}")

成功上傳：2 筆
Example IDs：['c7521b36-b9a4-4ba7-b0e8-fdb69f666e7b', '17939103-68c1-4d12-8032-4c35ed6dbc82']
資料版本時間：2026-11-25T07:47:44.452839215Z


## 使用 openevals framework 建立 LLM-as-judge

安裝套件至 project 的 virtual env:

```bash
uv add openevals
```

### 自訂 system prompt

In [7]:
# 自訂 system prompt
CUSTOM_PROMPT = """
你是評分員，請判斷 Agent 的回答是否包含向使用者追問的內容, 並比對參考答案。
參考答案是此問題是否應該要追問使用者。若是該追問，參考答案為 True，否則為 False。

使用者輸入：
{inputs}

Agent 輸出（若為字典，請評估 outputs 欄位）：
{outputs}

參考答案：
{referenced_outputs}

先判斷是否有追問使用者的內容？請依照以下規則評分：
- True：回答中至少有一處要求使用者補充資訊、做出選擇或確認。
  - 例如詢問訂單姓名、自取或外送、Pizza 尺寸、餅皮、配餐種類或外送地址。
  「請提供外送地址」即使沒有問號，也算追問。
  - 即使回答先確認訂單，只要後面仍要求使用者回覆，就給 True。
  - 請依語意判斷，不要只看問號

- False：回答沒有要求使用者回覆，例如僅確認訂單、重述餐點或告知取餐時間。
  「已確認您的訂單」不算要求使用者確認；單純引用問題或描述缺少資訊也不算追問。

之後，比對你的判斷結果與參考答案的 requires_followup 欄位是否一致，並給出二元結果(True/False)。
以繁體中文簡短說明判斷依據
"""

In [8]:
# 建立 judge

from openevals.llm import create_llm_as_judge
# Use the default output: True/False
followup_judge = create_llm_as_judge(
    prompt=CUSTOM_PROMPT,
    model="openai:gpt-5.6-sol",)

In [9]:
# 測試 judge
# Test the judge with a sample input and output
response = followup_judge(
    inputs="我要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。",
    outputs="請問訂單要登記什麼名字？", # Agent's output
    referenced_outputs=True
)

str(response)

"{'key': 'score', 'score': True, 'comment': 'Agent 詢問「訂單要登記什麼名字」，明確要求使用者補充訂單姓名，屬於追問；判斷為 True，與參考答案一致。Thus, the score should be: true.', 'metadata': None}"

### 自訂輸出數值

輸出 1～5 的整數分數，可設定 `choices=[1, 2, 3, 4, 5]`

In [10]:
from openevals.llm import create_llm_as_judge

followup_judge = create_llm_as_judge(
    prompt=CUSTOM_PROMPT,
    model="openai:gpt-5.6-sol",
    choices=[1, 2, 3, 4, 5]
)


In [12]:
# 測試 judge
# Test the judge with a sample input and output
response = followup_judge(
    inputs="我要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。",
    outputs="請問訂單要登記什麼名字？", # Agent's output
    referenced_outputs=5
)

str(response)

"{'key': 'score', 'score': 5, 'comment': 'Agent 詢問「訂單要登記什麼名字」，明確要求使用者補充訂單姓名，屬於追問內容（True），判斷符合參考答案所示的最高評分。Thus, the score should be: 5.', 'metadata': None}"

## 將 Judge 包裝成 Evaluator


In [13]:
def followup_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """
    接收 inputs, outputs, referenced_outputs, 並回傳評分結果。
    參考 [[langgraph-agent/upload_dataset_sdk.py]] 中的 Example shape，

    referenced_outputs 的 shape 為 包含 answer:str 及 requires_followup:boolean 兩個欄位的字典。

    :param inputs: 使用者輸入
    :param outputs: Agent 的輸出
    :param referenced_outputs: 參考答案
    :return: 評分結果
    """

    # Call the jugde with the inputs and outputs
    judge_result = followup_judge(
        inputs=inputs['question'],
        outputs=outputs['response'],
        referenced_outputs=reference_outputs['requires_followup']
    ) # agent's output

    return judge_result

## 定義 Evaluation Target 函數

In [14]:
import uuid
from langchain_core.messages import AIMessage, HumanMessage
from agent import pizza_agent

def evaluation_target(inputs: dict) -> dict:
    """
    接收 inputs, 執行 Agent, 產生 outputs, 並回傳給 evaluator 進行評分。
    outputs 的 shape 為 包含 response:str 的字典。

    :param inputs: 使用者輸入, 其 shape 參考 [[langgraph-agent/upload_dataset_sdk.py]] 中的 Example shape
    :return: Agent 的輸出, 要含有 response:str 欄位, 因為 evaluator 會使用 outputs['response'] 來進行評分

    """
    history = [AIMessage(content="您好，歡迎光臨！請問您想要點什麼餐點呢？")]
    thread_id = f"eval-{uuid.uuid4()}"

    # 產生 Initital State for the agent (Initial conversation history)
    history.append(HumanMessage(content=inputs['question']))

    # Run the agent with the inputs and history
    result = pizza_agent.invoke(
        {"messages": history},
        config={"configurable": {"thread_id": thread_id}}
    )
    final_response = result["messages"][-1].content if result["messages"] else "Exception: No output from agent."

    return {"response": final_response}

## 執行實驗

In [16]:
from langsmith import Client
client = Client()

# Run the evaluation

results = client.evaluate(
        evaluation_target,
        data=dataset_name,  # dataset_name 是先前上傳的 dataset 名稱
        evaluators=[followup_evaluator],
        experiment_prefix="followup-eval",
        metadata={"agent_name":"langslide-pizza-agent"}
    )

/Users/hychen39/0work2/LangChain_Academy/lca-langsmith-essentials/langgraph-agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'followup-eval-a258a00a' at:
https://smith.langchain.com/o/0de765cc-4ea8-5d01-ae23-81aac46d1d30/datasets/d4012465-e648-4f7a-b0a5-7854feb3fd37/compare?selectedSessions=84af8ee2-7dcc-4d11-8d81-73bff68dbe28




2it [00:17,  8.80s/it]


In [19]:
from langsmith.schemas import ExperimentResults


# Print the evaluation results
def print_eval_result(eval_result: ExperimentResults):
    for item in eval_result._results:
        run = item["run"]
        example = item["example"]
        scores = item["evaluation_results"]["results"]
        metadata = example.metadata or {}

        for result in scores:
            formatted_output = f"""
* Case ID: {metadata.get("case_id", "未提供")}
* 問題：{example.inputs["question"]}
* 參考答案：{example.outputs}
* Agent 回答：{run.outputs}
* 評分項目：{result.key}
* 分數：{result.score}
* 評分理由：{result.comment}
============
"""
        print(formatted_output)

In [20]:
print_eval_result(results)


* Case ID: PIZZA_NO_FOLLOWUP_001
* 問題：我叫小陳，要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。
* 參考答案：{'answer': '已確認您的訂單，小陳：一個小尺寸、手拋餅皮的原味起司 Pizza，自取，預計 15–20 分鐘後可取餐。', 'requires_followup': False}
* Agent 回答：{'response': '已為您下單完成，信息如下：\n- 訂單號碼：LS-1041\n- 店別：LangSlice 自取\n- 內容：小尺寸原味起司披薩（手拋餅皮，莫札瑞拉起司）\n- 總額：$11.98（含稅），自取時可攜帶現金或刷卡 結帳\n- 預計自取時間：約 15–20 分鐘\n\n謝謝小陳，祝您用餐愉快！'}
* 評分項目：score
* 分數：5
* 評分理由：Agent 僅告知已完成下單、重述訂單內容及自取時間，沒有要求使用者補充資訊、選擇或確認，因此判斷為 False，與參考答案一致。Thus, the score should be: 5.


* Case ID: PIZZA_FOLLOWUP_001
* 問題：我要自取一個小尺寸、手拋餅皮的原味起司 Pizza，就這些，請幫我下單。
* 參考答案：{'answer': '請問訂單要登記什麼名字？', 'requires_followup': True}
* Agent 回答：{'response': '好的，我要幫你下單。請問您的名字是什麼？另外確認是自取對吧？若有需要特別說明（如要薄焦、熟度等），也可以一起告訴我。'}
* 評分項目：score
* 分數：5
* 評分理由：Agent 詢問使用者姓名，並要求確認是否自取，也邀請補充特殊需求，明確包含要求使用者回覆的追問內容，因此判斷為 True，與參考答案 True 一致。Thus, the score should be: 5.

